# 05 — Win-probability curves

**Goal:** the graph this project set out to make. For any match, P(Team0 wins) at each moment of the game, like Valve's Dota 2 graph.

`python -m deadlock_predictor.viz.curve <match_id>` draws one. This notebook shows how it works, checks it, and looks at what the curves say.

**Two sets of models appear here:**
- **Deploy models** (`refit --all`): the final model (LightGBM + heroes) and the baseline, refit on **every** split. The curve tool uses them by default: the newest data, and every hero so far (including hero 87). Nothing is left to score them on, so their expected quality is the M4 test score, or the fresh matches in section 5.
- **M4 models** (`refit`): the same, fitted on train + validation only. For matches in the **test** split they are out of sample, so sections 3–4 use them to stay honest.

**Before running:** `python -m deadlock_predictor.models.refit --all` (about 6 minutes). Sections 2 and 5 need the internet (deadlock-api.com).

In [ ]:
import time
import warnings
from pathlib import Path

import duckdb
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from deadlock_predictor.data.dumps import USER_AGENT
from deadlock_predictor.data.match import api_rows, fetch_match, local_rows, register
from deadlock_predictor.features.heroes import build_heroes
from deadlock_predictor.features.snapshots import LABEL, build_snapshots, filter_matches
from deadlock_predictor.models.data import (
    SNAPSHOTS_PATH,
    load_split,
    predict_saved,
    snapshot_columns,
)
from deadlock_predictor.models.evaluate import scores
from deadlock_predictor.viz.curve import draw, load_models, match_features

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS = ROOT / "models"
deploy = load_models()  # trained on every split (python -m deadlock_predictor.models.refit --all)
m4 = {  # trained on train + validation only: never saw the test split
    "baseline": joblib.load(MODELS / "baseline_refit.joblib"),
    "final": joblib.load(MODELS / "gbm_heroes_refit.joblib"),
}
plt.rcParams["figure.figsize"] = (9, 4)
warnings.filterwarnings("ignore", message="The inplace keyword")
print({k: v["fit_on"] for k, v in deploy.items()})

## 1. From a match to a curve

1. **Rows:** the match's 12 player rows, from the local dump or, if it isn't there, from the API (`data.match.match_rows`).
2. **Features:** exactly the training code (`features.snapshots`, `features.heroes`), one row per stat snapshot. The final snapshot (taken as the game ends) is dropped as in training.
3. **Predictions:** each model gives P(Team0 wins) per row.

Snapshots exist only at 3, 6, 9, 12, 15 minutes and then every 5, so a curve has one point per snapshot, joined by straight lines. A tower falling at minute 13 shows up at the 15-minute point. Events are marked on the time axis so you can see what happened in between.

A test match's snapshots:

In [ ]:
match_id = int(load_split("test", []).match_id.iloc[0])
rows = local_rows(match_id)
snaps, heroes, failed = match_features(rows)
print(f"{len(rows)} player rows → {len(snaps)} snapshots; failed filters: {failed or 'none'}")
snaps[["t", "net_worth_diff", "kills_diff", "obj_tier1_diff", "obj_tier2_diff"]]

## 2. Does the API give the same features as the dump?

The API returns the same data with numeric codes: team 0/1 instead of `'Team0'`, objective id 10 instead of `'TitanShieldGenerator1'`. Also, the lobby rank isn't stored at all, so it's recomputed from the 12 players' ranks (`data.match.lobby_badge`). If any of that were wrong, curves for new matches would silently differ from what the model learned. So, for random matches present in both sources, the two paths are compared feature by feature:

In [ ]:
def features(rows):
    con = duckdb.connect()
    register(con, rows)
    filter_matches(con)
    snaps = build_snapshots(con).df().drop(columns="start_time")
    return snaps, build_heroes(con).df()


table = SNAPSHOTS_PATH.as_posix()
sample = duckdb.sql(
    f"SELECT DISTINCT match_id FROM '{table}' USING SAMPLE 15 ROWS (reservoir, 3)"
).fetchall()
identical = 0
for (mid,) in sample:
    s_dump, h_dump = features(local_rows(mid))
    s_api, h_api = features(api_rows(fetch_match(mid)))
    try:
        pd.testing.assert_frame_equal(s_dump, s_api, check_dtype=False)
        pd.testing.assert_frame_equal(h_dump, h_api, check_dtype=False)
        identical += 1
    except AssertionError as error:
        print(mid, str(error).splitlines()[0])
    time.sleep(0.3)
print(f"{identical} of {len(sample)} matches: identical features from the dump and the API")

## 3. What curves look like

Four test matches picked by shape, drawn with the **M4 models** (which never trained on test):
- **stomp:** the eventual winner never fell below 60%;
- **comeback:** the eventual winner fell below 15% at some point;
- **coin flip:** the final model stayed between 35% and 65% all game;
- **decided by the draft:** the baseline (which ignores heroes) starts near 50%, but the final model starts far from it.

The lower panel shows **what moved the final model's prediction** since the previous point, in logit units: the SHAP contributions from notebook 04, summed into groups (net worth, fights, levels & farm, objectives, mid-boss, heroes, time & rank). The first bar is the move from 50%.

In [ ]:
# Every test match, predicted by the M4 models (which never trained on test).
cols = sorted({c for s in m4.values() for c in snapshot_columns(s["columns"])})
test = load_split("test", cols)
test["p"] = predict_saved(m4["final"], test)
test["p_baseline"] = predict_saved(m4["baseline"], test)
won = test[LABEL].to_numpy()
test["p_winner"] = np.where(won, test.p, 1 - test.p)  # P(eventual winner) at each snapshot
shape = test.groupby("match_id").agg(
    lowest=("p_winner", "min"),
    last=("p_winner", "last"),
    p_min=("p", "min"),
    p_max=("p", "max"),
    first_final=("p", "first"),
    first_baseline=("p_baseline", "first"),
    snapshots=("t", "size"),
)
shape = shape[shape.snapshots >= 7]
draft = (shape.first_final - 0.5).abs()[(shape.first_baseline - 0.5).abs() < 0.05]
picks = {
    "stomp": shape[shape.lowest > 0.6].sort_values("last").index[-1],
    "comeback": shape[shape.lowest < 0.15].sort_values("snapshots").index[-1],
    "coin flip": shape[(shape.p_min > 0.35) & (shape.p_max < 0.65)]
    .sort_values("snapshots")
    .index[-1],
    "decided by the draft": draft.idxmax(),
}
picks

In [ ]:
for name, mid in picks.items():
    fig, curve, events = draw(int(mid), models=m4)
    fig.suptitle(name, x=0.02, ha="left", fontsize=12, fontweight="bold")
    plt.show()

How to read them:
- **The first point is mostly the draft.** At 3 minutes stats barely differ, so the heroes bar dominates the first move. In the "decided by the draft" match it is worth about −2.4 logit (≈ 8% for Team0) before anything has happened. The baseline, which can't see heroes, even had Team0 at 84% by 30 minutes, but the draft favourite won.
- **Net worth drives the middle; objectives the end.** Late moves are mostly red (objectives): shield generators and the Titan falling are what swing a 40-minute game.
- **The comeback** is a long, chaotic game: one team took the other's Titan around 45 minutes and the curve swung hard, then mid-bosses and late buildings swung it back. Long games are where the model is least sure (notebook 04's log loss by time rises after 35 minutes).
- **The final model and the baseline mostly agree** once stats diverge; they differ early, where only the final model knows the heroes.

## 4. Are the curves honest about comebacks?

A curve that says 10% should be right about comebacks: of all teams it ever puts at 10% or less, roughly that share should still win. Not exactly 10%, though. When a curve first drops below 10%, it often lands well below (say at 4%), so the fair comparison is with the probability **at the first snapshot where it dropped to the threshold**. That's the "expected" column: the average P(win) at that moment, which is what a calibrated model promises. (This works because a calibrated probability is a fair bet at every moment, wherever you choose to look; statisticians call this a *martingale*.)

Computed on every test match with the M4 models:

In [ ]:
def comeback_table(df, column, thresholds=(0.3, 0.2, 0.1, 0.05)):
    """For each threshold: teams whose P(win) fell to it at some snapshot, how often they won,
    and the average P at the first snapshot where it did (what a calibrated model expects)."""
    rows = {}
    df = df.sort_values(["match_id", "t"])
    for sign in (1, -1):  # Team0's view, then Team1's
        p_team = df[column] if sign == 1 else 1 - df[column]
        team_won = df[LABEL] if sign == 1 else ~df[LABEL]
        for th in thresholds:
            below = df.assign(p_team=p_team, team_won=team_won)[p_team <= th]
            first = below.groupby("match_id").first()
            agg = rows.setdefault(th, {"teams": 0, "won": 0, "expected": 0.0})
            agg["teams"] += len(first)
            agg["won"] += int(first.team_won.sum())
            agg["expected"] += float(first.p_team.sum())
    table = pd.DataFrame(rows).T
    table["win rate"] = table.won / table.teams
    table["expected"] = table.expected / table.teams
    table.index.name = "P(win) fell to"
    return table[["teams", "win rate", "expected"]]


pd.concat(
    {"final model": comeback_table(test, "p"), "baseline": comeback_table(test, "p_baseline")},
    axis=1,
).round(3)

- **The curves are honest about comebacks.** Teams that fell to 10% or less won 5.2% of the time, against 5.4% expected; at 5%: 2.0% vs 2.4%.
- **At 20–30% the final model is slightly too pessimistic** (13.6% won vs 12.9% expected at 20%). That's the overconfidence notebook 04 found: hero effects learned on older data are a bit too strong.
- The final model puts **more** teams below each threshold than the baseline does. That's what a sharper model looks like: it's surer earlier, and still calibrated.
- Comebacks are rare but real: about 1 in 20 teams written off at 10% won anyway.

## 5. Matches played after the data ends

The dump ends on Oct 24. The API's `recently-fetched` list has matches from the last day or two. **No model has seen these**: they're a small, fresh test set for the deploy models, and for the M4 models to compare against. We score them with the same filters as training (complete 6v6 teams, no abandons). The list changes over time, so rerunning gives different matches.

In [ ]:
listing = requests.get(
    "https://api.deadlock-api.com/v1/matches/recently-fetched",
    headers={"User-Agent": USER_AGENT},
    timeout=30,
).json()
last_local = int(duckdb.sql(f"SELECT max(match_id) FROM '{table}'").fetchone()[0])
new_ids = sorted(
    m["match_id"]
    for m in listing
    if m["match_mode"] == 4 and m["game_mode"] == 1 and m["match_id"] > last_local
)
print(f"{len(new_ids)} ranked 6v6 matches newer than the dump (newest local id {last_local})")

In [ ]:
fresh_snaps, fresh_heroes, skipped = [], [], 0
for mid in new_ids[:200]:
    try:
        rows = api_rows(fetch_match(mid))
    except requests.RequestException:
        skipped += 1
        continue
    snaps, heroes, failed = match_features(rows)
    if failed:  # same filters as training, so the score is comparable
        skipped += 1
        continue
    fresh_snaps.append(snaps)
    fresh_heroes.append(heroes)
    time.sleep(0.2)
fresh = pd.concat(fresh_snaps, ignore_index=True)
fresh_heroes = pd.concat(fresh_heroes, ignore_index=True)
y = fresh[LABEL].to_numpy()
fresh_scores = pd.DataFrame(
    {
        f"{name} ({version})": scores(y, predict_saved(models[name], fresh, fresh_heroes))
        for version, models in (("deploy", deploy), ("M4", m4))
        for name in ("baseline", "final")
    }
).T
print(f"{fresh.match_id.nunique()} matches, {len(fresh)} snapshots; skipped {skipped}")
fresh_scores.round(4)

The recent-matches list changes from hour to hour, so each run scores a different ~150–180 matches. Across three runs on Oct 25:

| run | matches | baseline (deploy) | final (M4) | **final (deploy)** |
|---|---|---|---|---|
| 1 | 145 | 0.578 | 0.549 | **0.535** |
| 2 | 176 | 0.587 | 0.529 | **0.526** |
| 3 | 167 | 0.549 | 0.498 | **0.496** |

- **The final model beats the baseline by 0.04–0.06 every time**, on games no decision ever touched. That's at least as large as on the test split (0.024).
- **The deploy version beat the M4 version in every run, by 0.001 to 0.014.** It has seen hero 87 and newer hero strengths. The baselines (no heroes) score the same either way, so the difference comes from heroes being up to date. The size of the gain is uncertain with samples this small; its direction was consistent.
- **Absolute scores swing a lot between runs** (0.50–0.59 for the same model): ~150 matches is a small sample, and the mix of games differs. That's why we compare models *on the same matches*, and why ECE on ~1,500 rows (0.03–0.05) is mostly noise. Section 4's comeback table, on 60,000 matches, is the reliable calibration check.

The lesson from notebook 04 holds: **for heroes, recent data matters.** A deployed version would retrain regularly.

Three of them:

In [ ]:
for mid in fresh.match_id.unique()[:3]:
    draw(int(mid), models=deploy)
    plt.show()

## 6. Summary

- `python -m deadlock_predictor.viz.curve <match_id>` draws the curve for **any** ranked match: from the local dump, or from the API for new ones. The API and dump paths give identical features (15 of 15 sampled matches).
- Each point is a real stat snapshot (every 3–5 minutes); events in between are marked on the time axis; the lower panel says what moved the prediction.
- The curves are **calibrated where it matters for a viewer**: a team shown at 10% wins about 5% of the time from there, as a calibrated curve promises.
- On fresh matches after the data ends, the deploy model beats both the baseline and the older M4 model: **retraining with recent matches is part of the model**, not an afterthought.

Ideas for later (Milestone 6): retrain on a schedule; finer curves from the API's per-second position and health data; a dashboard or live predictions.